In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.models as models
import torch.optim.lr_scheduler as lr_scheduler

import time
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange, tqdm

In [ ]:
batch_size = 32
num_epochs = 20
learning_rate = 1e-3   # sets the inital learning rate to 0.001
#start_epoch = 0
unfreeze_eight = 5    # unfreeze the 8th block of the EfficentNet B5 Architecture
unfreeze_seven = 10   # unfreezes the 7th block of the EfficentNet Architecture
unfreeze_six = 15   # unfreezes the 6th block of the EfficentNet Architecture
#learning_rate_epoch_five = 5   # reduces the learning rate at the 5th epoch
img_size = (456, 456)   # 456*456px input size for EfficentNet B5

# imports and unzips the dataset locally for faster processing or load data from google drive folder
from google.colab import drive
#data_root = '/content/drive/MyDrive/Version 1 DATASET/Final Dataset'
!unzip -q '/content/drive/MyDrive/Version 1 DATASET/Final Dataset.zip' -d '/content/local_data'
data_root = '/content/local_data/Final Dataset - Copy'

In [ ]:
start_from_checkpoint = False

model_name = 'v1DataEfficentNetB5Model_v12' # change each time I train a new model

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")   # train on the GPU cuda cores

In [ ]:
transform = transforms.Compose([transforms.Resize(img_size),    # sets images to a square 456x456 resolution and load them into tensors
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225])])

In [ ]:
train_examples = datasets.ImageFolder(root=f'{data_root}/train', transform=transform)   # pulls the data from their respective folders
val_examples = datasets.ImageFolder(root=f'{data_root}/val', transform=transform)
test_examples = datasets.ImageFolder(root=f'{data_root}/test', transform=transform)

num_classes = len(train_examples.classes)
print("Classes:", train_examples.classes)

train_loader = DataLoader(train_examples, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)   # loads the data into the script
val_loader = DataLoader(val_examples, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_examples, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

Classes: ['ResizedBad', 'ResizedGood']


In [ ]:
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights   # loads the EfficentNet B5 model architecture in with the deafault weights (IMAGENET1K_V1)
weights = EfficientNet_B5_Weights.DEFAULT
model = efficientnet_b5(weights=weights)

for param in model.parameters():    # initally freezes all 8 blocks of the model architecture
    param.requires_grad = False

in_features = model.classifier[1].in_features   # creates a classifier output layer for 2 classes (ResiszedGood & ResizedBad)
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)
print(f"model loaded for {num_classes} classes\n")    # check to make sure the model is loaded for the 2 classes
#print(model)

criterion = nn.CrossEntropyLoss()   # defines how loss is calculated
optimizer = optim.Adam(model.parameters(), lr=learning_rate)    # initalizes Adam as the optimizer

Downloading: "https://download.pytorch.org/models/efficientnet_b5_lukemelas-1a07897c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b5_lukemelas-1a07897c.pth


100%|██████████| 117M/117M [00:00<00:00, 136MB/s]


Model loaded for 2 classes

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
            (1): BatchNorm2d(48, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )


In [ ]:
#scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)    # lowers the learning rate if the validation loss does not decrease after a span of 3 epochs

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs):   # add scheduler if reduce learning rate on plateau is needed
    start_time = time.time()
    best_val_loss = float('inf')
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()    # sets model to train on the training dataset
        train_loss = 0.0    # initalizes variables for the calculation of the training accuracy and loss
        train_correct = 0
        train_total = 0

        if epoch == learning_rate_epoch_five:   # reduces learning rate at the 5th epoch
           for param_group in optimizer.param_groups:
               param_group["lr"] = param_group["lr"] * 0.1
               new_lr = param_group["lr"]

           print(f"Epoch [{epoch}/{num_epochs}], Learning Rate: {new_lr:.6f}")

        if epoch == unfreeze_eight:   # unfreezes the 8th block of the EfficentNet B5 architecture to allow for fine tuning
            for param in model.features[8:].parameters():
                param.requires_grad = True

                for param_group in optimizer.param_groups:    # sets the learning rate to 0.0001
                    param_group["lr"] = 1e-4
                    new_lr = param_group["lr"]

            print(f"Block 8 has been unfrozen")
            print(f"Epoch [{epoch}/{num_epochs}], Learning Rate: {new_lr:.6f}")

        if epoch == unfreeze_seven:   # unfreezes the 7th block of the EfficentNet B5 architecture to allow for further fine tuning
            for param in model.features[7:].parameters():
                param.requires_grad = True

                for param_group in optimizer.param_groups:    # lowers the learning rate to 0.00001
                    param_group["lr"] = 1e-5
                    new_lr = param_group["lr"]

            print(f"Block 7 has been unfrozen")
            print(f"Epoch [{epoch}/{num_epochs}], Learning Rate: {new_lr:.6f}")

        if epoch == unfreeze_six:   # unfreezes the 6th block of the EfficentNet B5 architecture to allow for even further fine tuning
            for param in model.features[6:].parameters():
                param.requires_grad = True

                for param_group in optimizer.param_groups:    # lowers the learning rate to 0.000001
                    param_group["lr"] = 1e-6
                    new_lr = param_group["lr"]

            print(f"Block 6 has been unfrozen")
            print(f"Epoch [{epoch}/{num_epochs}], Learning Rate: {new_lr:.6f}")

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")   # creates a progress bar with the per epoch training progess

        for inputs, labels in pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
            predicted = torch.argmax(outputs, dim=1)
            train_correct += predicted.eq(labels).sum().item()
            train_total += labels.size(0)
            avg_train_acc = 100. * train_correct / train_total
            pbar.set_postfix(loss=f"{loss.item():.4f}", train=f"{avg_train_acc:.2f}%")

        avg_train_loss = train_loss / len(train_loader.dataset)   # calculates training accuracy and loss
        train_acc = 100. * train_correct / train_total

        model.eval()    # switches model to evaluation mode to test its performance on the validation dataset
        val_loss = 0.0    # initalizes varibles for the calculation of the validation loss and accuracy
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                predicted = torch.argmax(outputs, dim=1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)
                avg_val_acc = 100. * val_correct / val_total
                pbar.set_postfix(loss=f"{loss.item():.4f}", val=f"{avg_val_acc:.2f}%")

        avg_val_loss = val_loss / len(val_loader.dataset)    # calculate the validation accuracy and loss
        val_acc = 100. * val_correct / val_total

        #scheduler.step(avg_val_loss)   # allows scheduler to run off of the validation loss data

        train_losses.append(avg_train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1}:\n")    # outputs the progress / results after each epoch during training
        print(f"Training Accuracy: {train_acc:.4f} | Training Loss: {avg_train_loss:.4f} | Validation Accuracy: {val_acc:.4f} | Validation Loss: {avg_val_loss:.4f}:\n")

        if avg_val_loss < best_val_loss: # saves model weights if loss is decreasing
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f'{model_name}.pt')
            print("Checkpoint Saved!")

    plt.figure(figsize=(12, 5)) # creates a plot to house the accuracy and loss plots

    plt.subplot(1, 2, 1)    # creates a subplot in the main plot to graph the training and validation accuracy after each epoch of training
    plt.plot(train_accuracies, label='Train Accuracy')
    plt.plot(val_accuracies, label='Val Accuracy')
    plt.title('Accuracy History')
    plt.xlabel('Epoch')
    plt.ylabel('Percentage')
    plt.legend()

    plt.subplot(1, 2, 2)    # creates a subplot in the main plot to graph the training and validation loss after each epoch of training
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title('Loss History')
    plt.xlabel('Epoch')
    plt.ylabel('Loss Value')
    plt.legend()

    plt.tight_layout()
    plt.show()

Remember to download the model weights after it has been trained!